# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AL-NAJAF/MLAssgn01/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import os, sys, subprocess

REPO_URL = "https://github.com/AL-NAJAF/MLAssgn01"
REPO_DIR = "MLAssgn01"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())
print("Contents:", os.listdir("."))

Now in: /content/MLAssgn01/MLAssgn01
Contents: ['CLAUDE.md', '.github', 'docs', 'GUIDE.md', 'README.md', '.gitignore', 'skills', 'work', 'DATA_USE.md', '.git', 'requirements.txt', 'AGENTS.md', 'scripts', 'outputs', 'notebooks', 'LICENSE', 'data', 'submission', 'SETUP.md']


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing Lane 2: Refresh / Content Opportunity Scoring. I picked this because I already
explored this exact problem in the Week 1 and Week 2 starter notebooks — comparing a hand-written
rule against a learned model to flag pages that are declining and worth reviewing. The starter
data showed a random forest beating a simple hand rule by a wide margin (Precision@50 of 0.740
vs 0.240), which suggests there's real, learnable signal in this data beyond what a simple rule
can capture. I want to build on that with a stronger, future-looking label using the full
warehouse dataset instead of the small proxy label used in the starter notebook.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision this improves: which pages an editor should review first for refresh, out of
thousands, given limited time.

Who acts on it: a content editor or SEO strategist — they get a ranked list and start from
the top.

Cost of a wrong call: if we flag a page that wasn't actually a priority, the editor wastes
time reviewing it (a small, recoverable cost). If we miss a page that was genuinely declining,
it keeps losing traffic silently until someone notices another way (a bigger, compounding
cost). So false negatives are more expensive than false positives here.

Why ML, not just a rule: a simple hand-written rule (e.g. "stale AND still visible") already
captures some of this, and we already tested it in the starter notebooks. But the Week 1/2
results showed a learned model catching real signal the rule missed (Precision@50 jumped from
0.240 to 0.740) — meaning the pattern is real, but too tangled across many signals for one
if-statement to capture on its own.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [16]:
import os
print("Current directory:", os.getcwd())
print("Contents here:", os.listdir("."))

Current directory: /content/MLAssgn01/MLAssgn01
Contents here: ['CLAUDE.md', '.github', 'docs', 'GUIDE.md', 'README.md', '.gitignore', 'skills', 'work', 'DATA_USE.md', '.git', 'requirements.txt', 'AGENTS.md', 'scripts', 'outputs', 'notebooks', 'LICENSE', 'data', 'submission', 'SETUP.md']


In [17]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# NOTE: ctr is already a percentage (e.g. 0.76 = 0.76%, not 76%) -- read as-is, don't divide by 100 again
# NOTE: avg_position == 0 means "no data", not rank zero -- exclude those rows from position stats
valid_position = df[df["avg_position"] > 0]

declining_rate = (df["trend_direction"].str.lower() == "down").mean()
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
avg_ctr_valid_position = valid_position["ctr"].mean()

print(f"Total pages: {df.shape[0]}")
print(f"Share currently labeled 'declining': {declining_rate:.3f}")
print(f"Stale (180+ days) AND still visible (500+ impressions): {stale_visible} pages")
print(f"Average CTR (%) among pages with a real position, excluding avg_position=0: {avg_ctr_valid_position:.3f}")

Total pages: 30000
Share currently labeled 'declining': 0.542
Stale (180+ days) AND still visible (500+ impressions): 17 pages
Average CTR (%) among pages with a real position, excluding avg_position=0: 0.520


## 3. Quick look at the data (2-3 real numbers)

In the starter dataset (30,000 pages), about 54.2% are currently labeled as "declining" —
a large, realistic pool of candidates for a refresh-scoring system. Only 17 pages are both
stale (180+ days since last update) AND still getting strong visibility (500+ impressions) —
a small but meaningful group of high-value, low-effort refresh candidates: pages clearly worth
revisiting that are easy for a reviewer to spot quickly. Among pages with a real recorded
position (excluding avg_position = 0, which means "no data"), the average CTR is 0.520% —
consistent with the Week 1 finding that click-through rates are generally low and highly
position-dependent, which is exactly the kind of signal a refresh-scoring model would need to
weigh carefully. Together with the Week 1/2 result that a random forest reached Precision@50
of 0.740 versus 0.240 for a simple hand rule, this supports Lane 2 as a lane with real,
learnable signal worth pursuing over the next 7 weeks.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim: a ranked, evidence-backed list of pages likely worth reviewing, based on
observed signals like staleness, visibility, and position -- never using trend_direction or
trend_pct as inputs, since those are literally what the "declining" label is built from (using
them as features would just teach the model to copy the label, not find real signal).

What I can't claim: that refreshing a flagged page will cause it to recover (that needs a real
experiment, not historical data), or that this reflects Google's actual ranking algorithm.
Results are observed and directional -- decision-support for a human reviewer, not a guarantee.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.